# Costruire il Primo Agente AI con `smolagents`

In questo notebook costruiremo un agente AI passo dopo passo.

Un **agente AI** è un sistema che:
1. Riceve un obiettivo in linguaggio naturale
2. **Ragiona** su come raggiungerlo (Think)
3. Sceglie e **esegue azioni** tramite tool (Act)
4. **Osserva** il risultato (Observe)
5. Ripete il ciclo finché non raggiunge la risposta finale

```
┌─────────────────────────────────────────┐
│            CICLO DELL'AGENTE            │
│                                         │
│   Obiettivo → THINK → ACT → OBSERVE     │
│                  ↑_____________│        |
└─────────────────────────────────────────┘
```

Useremo **`smolagents`**, la libreria di HuggingFace per costruire agenti in modo semplice.

## 1. Installazione delle Dipendenze

Se stai usando l'ambiente `uv` di questo progetto, le dipendenze sono già installate.

Se invece vuoi installare in un ambiente Colab o diverso, esegui la cella qui sotto (rimuovi il `#`):

In [ ]:
# Decommentare solo se NON si usa l'ambiente uv del progetto
# %pip install smolagents gradio pytz requests pyyaml python-dotenv duckduckgo-search huggingface_hub

## 2. Import delle Librerie

Importiamo tutto quello che ci serve:
- `smolagents`: il framework per gli agenti
- `dotenv`: per caricare il token HF da un file `.env` senza mai esporlo nel codice
- `datetime`, `pytz`: per lavorare con date e fusi orari

In [1]:
import os
import datetime
import pytz

from dotenv import load_dotenv
from smolagents import (
    CodeAgent,           # Il tipo di agente che scrive ed esegue codice Python
    InferenceClientModel, # Il modello LLM servito via API HuggingFace
    DuckDuckGoSearchTool, # Tool built-in per cercare su internet
    tool,                # Decoratore per trasformare funzioni in tool
)

print("Librerie importate correttamente!")

Librerie importate correttamente!


## 3. Autenticazione HuggingFace

Per usare i modelli via API di HuggingFace è necessario un token.

**Come ottenere il token:**
1. Vai su https://hf.co/settings/tokens
2. Crea un nuovo token con permessi `write`
3. Copia il file `.env.example` in `.env` nella cartella del progetto
4. Inserisci il tuo token: `HF_TOKEN=hf_tuoTokenQui`

> **Sicurezza:** il file `.env` è elencato nel `.gitignore`, quindi non verrà mai committato su Git. Non condividere mai il tuo token!

In [2]:
# Carica le variabili d'ambiente dal file .env
load_dotenv()

hf_token = os.getenv("HF_TOKEN")

if hf_token:
    # Mostriamo solo i primi 8 caratteri per conferma, mai il token completo
    print(f"Token HF caricato correttamente: {hf_token[:8]}...")
else:
    print("ATTENZIONE: HF_TOKEN non trovato!")
    print("Crea un file .env con: HF_TOKEN=hf_tuoTokenQui")

Token HF caricato correttamente: hf_GxbIC...


## 4. I Tool: Le "Mani" dell'Agente

I **tool** sono funzioni Python che l'agente può chiamare per interagire con il mondo esterno.

Per creare un tool valido con `smolagents` bisogna:
1. Decorare la funzione con `@tool`
2. **Specificare i tipi** degli argomenti e del valore di ritorno (type hints)
3. Scrivere un **docstring chiaro** con la descrizione della funzione e di ogni argomento

Il docstring è fondamentale: l'LLM lo legge per capire *quando* e *come* usare il tool.

In [3]:
# ----- TOOL 1: Ora locale in un fuso orario -----

@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """Restituisce l'ora locale corrente in un fuso orario specificato.
    Args:
        timezone: Una stringa che rappresenta un fuso orario valido (es. 'Europe/Rome', 'America/New_York').
    """
    try:
        tz = pytz.timezone(timezone)
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"L'ora locale in {timezone} è: {local_time}"
    except Exception as e:
        return f"Errore per il fuso orario '{timezone}': {str(e)}"


# Proviamo il tool direttamente (senza agente) per verificare che funzioni
print(get_current_time_in_timezone("Europe/Rome"))
print(get_current_time_in_timezone("America/New_York"))

L'ora locale in Europe/Rome è: 2026-05-10 18:22:48
L'ora locale in America/New_York è: 2026-05-10 12:22:48


In [4]:
# ----- TOOL 2: Calcolatrice semplice -----

@tool
def calculate(expression: str) -> str:
    """Calcola il risultato di un'espressione matematica testuale.
    Args:
        expression: L'espressione matematica da calcolare (es. '2 + 2', '15 * 4', '100 / 3').
    """
    try:
        # eval è sicuro qui perché limitiamo le operazioni matematiche
        allowed = set("0123456789+-*/()., ")
        if not all(c in allowed for c in expression):
            return "Errore: solo operazioni matematiche base sono permesse."
        result = eval(expression)
        return f"Risultato di '{expression}' = {result}"
    except Exception as e:
        return f"Errore nel calcolo: {str(e)}"


# Test del tool
print(calculate("(15 * 4) + 20"))
print(calculate("100 / 3"))

Risultato di '(15 * 4) + 20' = 80
Risultato di '100 / 3' = 33.333333333333336


## 5. Tool Built-in di smolagents

`smolagents` include già dei tool pronti all'uso. Il più utile è `DuckDuckGoSearchTool`, che permette all'agente di cercare informazioni su Internet.

Vediamo la lista dei tool che useremo:
| Tool | Descrizione |
|------|-------------|
| `DuckDuckGoSearchTool` | Cerca sul web via DuckDuckGo |
| `get_current_time_in_timezone` | Ora in un fuso orario (custom) |
| `calculate` | Calcolatrice (custom) |

In [5]:
# Istanziamo il tool di ricerca web built-in
search_tool = DuckDuckGoSearchTool()

# Testiamo anche questo direttamente
result = search_tool("HuggingFace smolagents framework")
print(result[:500], "...")  # Mostriamo solo i primi 500 caratteri

## Search Results

[smolagents · Hugging Face](https://huggingface.co/docs/smolagents/index)
smolagents is an open-source Python library designed to make it extremely easy to build and run agents using just a few lines of code. Key features of smolagents include

[Smolagents : Huggingface AI Agent Framework](https://smolagents.org/)
Smolagents is a minimalist AI agent framework developed by the Hugging Face team, crafted to enable developers to deploy robust agents with just a few lines of code. ...


## 6. Il Modello LLM

L'**LLM** è il "cervello" dell'agente: riceve il contesto (obiettivo + risultati dei tool) e decide cosa fare.

Usiamo `InferenceClientModel` di smolagents, che si connette all'API serverless di HuggingFace.

Il modello scelto è **`Qwen/Qwen2.5-Coder-32B-Instruct`**: ottimo per ragionare e scrivere codice.

In [6]:
# Creiamo il modello LLM che userà l'API di HuggingFace
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=hf_token,          # Il token che abbiamo caricato dal .env
    max_tokens=2048,
    temperature=0.5,          # 0 = deterministico, 1 = creativo
)

print(f"Modello pronto: {model.model_id}")

Modello pronto: Qwen/Qwen2.5-Coder-32B-Instruct


## 7. Creare l'Agente

Ora assembliamo tutto in un `CodeAgent`.

**Cos'è un CodeAgent?**  
È un tipo di agente che esprime le sue azioni scrivendo **blocchi di codice Python**. Invece di chiamare i tool come JSON, il modello scrive codice che viene eseguito nel runtime. Questo lo rende molto potente per task computazionali.

**Parametri chiave:**
- `model`: l'LLM da usare
- `tools`: lista di tool disponibili
- `max_steps`: numero massimo di passi Think→Act→Observe prima di fermarsi

In [7]:
# Raccogliamo tutti i tool in una lista
tools = [
    search_tool,                    # Ricerca web
    get_current_time_in_timezone,   # Ora nel mondo
    calculate,                      # Calcolatrice
]

# Creiamo il CodeAgent
agent = CodeAgent(
    model=model,
    tools=tools,
    max_steps=5,        # Max 5 cicli Think→Act→Observe
    verbosity_level=2,  # 0=silenzioso, 1=normale, 2=dettagliato
)

print("Agente creato!")
print(f"Tool disponibili: {[t.name for t in agent.tools.values()]}")

Agente creato!
Tool disponibili: ['web_search', 'get_current_time_in_timezone', 'calculate', 'final_answer']


## 8. Eseguiamo l'Agente!

Usiamo `agent.run()` per dare un obiettivo all'agente in linguaggio naturale.

Osserva il **ciclo Think → Act → Observe** nell'output:
- **Thinking**: il modello ragiona sul problema
- **Calling tool**: il modello esegue un'azione
- **Observation**: il risultato del tool viene letto

### Query 1: Fuso orario

In [8]:
# Domanda semplice: richiede un solo tool
risposta = agent.run(
    "Che ore sono adesso a Tokyo e a Los Angeles? Qual è la differenza?"
)

print("\n" + "="*50)
print("RISPOSTA FINALE:")
print(risposta)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Che ore sono adesso a Tokyo e a Los Angeles? Qual è la differenza?                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I need to find the current time in Tokyo and Los Angeles, then calculate the difference between them.              
                                                                                                                   
First, let me get the current time in Tokyo:                                                                       
<code>                                                                                                             
tokyo_time = get_current_time_in_timezone("Asia/Tokyo")                                                            
print(f"Current time in Tokyo: {tokyo_time}")                                                                      
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tokyo_time = get_current_time_in_timezone("Asia/Tokyo")                                                          
  print(f"Current time in Tokyo: {tokyo_time}")                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current time in Tokyo: L'ora locale in Asia/Tokyo è: 2026-05-11 01:23:15

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Based on the information I've gathered:                                                                            
                                                                                                                   
- Current time in Tokyo: 2026-05-11 01:23:15                                                                       
- Current time in Los Angeles: 2026-05-10 09:23:16                                                                 
                                                                                                                   
The difference between Tokyo and Los Angeles is 16 hours, with Tokyo being ahead.                                  
                                                                                                                   
Let me finalize the answer:                                                                                        
<code>                                                                                                             
final_answer("Current time in Tokyo is 2026-05-11 01:23:15 and in Los Angeles is 2026-05-10 09:23:16. The          
difference is 16 hours, with Tokyo being ahead.")                                                                  
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Current time in Tokyo is 2026-05-11 01:23:15 and in Los Angeles is 2026-05-10 09:23:16. The        
  difference is 16 hours, with Tokyo being ahead.")                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Current time in Tokyo is 2026-05-11 01:23:15 and in Los Angeles is 2026-05-10 09:23:16. The 
difference is 16 hours, with Tokyo being ahead.

[Step 4: Duration 1.78 seconds| Input tokens: 10,312 | Output tokens: 478]


RISPOSTA FINALE:
Current time in Tokyo is 2026-05-11 01:23:15 and in Los Angeles is 2026-05-10 09:23:16. The difference is 16 hours, with Tokyo being ahead.


### Query 2: Ricerca Web + Calcolo

Questa query richiede che l'agente usi più tool in sequenza: prima cerca su internet, poi eventualmente calcola.

In [9]:
# Domanda che combina ricerca e ragionamento
risposta = agent.run(
    "Cerca quanti parametri ha il modello Qwen2.5-Coder-32B e calcola quanta memoria "
    "servirebbe per caricarlo in float16 (2 byte per parametro). Rispondi in GB."
)

print("\n" + "="*50)
print("RISPOSTA FINALE:")
print(risposta)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Cerca quanti parametri ha il modello Qwen2.5-Coder-32B e calcola quanta memoria servirebbe per caricarlo in     │
│ float16 (2 byte per parametro). Rispondi in GB.                                                                 │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Per rispondere a questa domanda, devo prima cercare quanti parametri ha il modello Qwen2.5-Coder-32B. Poi,
una volta ottenuto questo numero, posso calcolare la memoria necessaria per caricarlo in float16, che richiede 2   
byte per parametro. Infine, convertirò il risultato da byte a gigabyte.                                            
                                                                                                                   
Inizierò cercando le informazioni sul modello Qwen2.5-Coder-32B.                                                   
<code>                                                                                                             
model_info = web_search("Qwen2.5-Coder-32B parameters")                                                            
print(model_info)                                                                                                  
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  model_info = web_search("Qwen2.5-Coder-32B parameters")                                                          
  print(model_info)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Qwen/Qwen2.5-Coder-32B-Instruct · Hugging Face](https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct)
from transformers import AutoModelForCausalLM, AutoTokenizer model_name = "Qwen/Qwen2.5-Coder-32B-Instruct" model =
AutoModelForCausalLM.from_pretrained( model_name, torch_dtype="auto", device_map="auto" ) tokenizer = 
AutoTokenizer.from_pretrained(model_name) prompt = "write a quick sort algorithm." messages = [ {"role": "system", 
"content": "You are Qwen, created by Alibaba Cloud.

[Qwen/Qwen2.5-Coder-32B · Hugging Face](https://huggingface.co/Qwen/Qwen2.5-Coder-32B)
As of now, Qwen2.5-Coder has covered six mainstream model sizes, 0.5, 1.5, 3, 7, 14, 32 billion parameters, to meet
the needs of different developers. Qwen2.5-Coder brings the following improvements upon CodeQwen1.5: Significantly 
improvements ...

[qwen2.5-coder-32b-instruct (Qwen) · Cloudflare AI docs · Cloudflare Workers AI 
docs](https://developers.cloudflare.com/workers-ai/models/qwen2.5-coder-32b-instruct/)
As of now, Qwen2.5-Coder has covered six mainstream model sizes, 0.5, 1.5, 3, 7, 14, 32 billion parameters, to meet
the needs of different developers. Qwen2.5-Coder brings the following improvements upon CodeQwen1.5: Try out this 
model with ...

[qwen2.5-coder](https://ollama.com/library/qwen2.5-coder)
Code Reasoning: Code reasoning refers to the model’s ability to learn the process of code execution and accurately 
predict the model’s inputs and outputs. The recently released Qwen2.5 Coder 7B Instruct has already shown 
impressive performance in code reasoning, and this 32B model takes it a step further.

[Qwen/Qwen2.5-Coder-32B-Instruct - Demo - DeepInfra](https://deepinfra.com/Qwen/Qwen2.5-Coder-32B-Instruct)
As of now, Qwen2.5-Coder has covered six mainstream model sizes, 0.5, 1.5, 3, 7, 14, 32 billion parameters, to meet
the needs of different developers. Qwen2.5-Coder brings the following improvements upon CodeQwen1.5: Significantly 
improvements ...

[qwen2.5-coder-32b-instruct Model by Qwen](https://build.nvidia.com/qwen/qwen2_5-coder-32b-instruct/modelcard)
⌘KCtrl+K · Login · Terms of Use · Your Privacy Choices · Contact · Copyright © 2026 NVIDIA Corporation · 
qwen2.5-coder-32b-instruct Model by Qwen | NVIDIA NIM · <img 
src="https://build.nvidia.com/akam/13/pixel_7a476816?a=dD1hYjRjYjM0NzlhYzNhMjdmNDU5YzdiMWQ2ZTgyMWI4M2Q5NGI4MjRjJmp.
..

[bartowski/Qwen2.5-Coder-32B-Instruct-GGUF · Hugging 
Face](https://huggingface.co/bartowski/Qwen2.5-Coder-32B-Instruct-GGUF)
Qwen/Qwen2.5-Coder-32B-Instruct · Quantized (119) this model · Collection · This collection contains some of the 
recent models larger than ~25B parameters that should be high quality and reliable • 15 items • Updated Nov 27, 
2024 • 57

[qwen2.5-coder:32b](https://ollama.com/library/qwen2.5-coder:32b)
Qwen 2.5 Coder series of models are now updated in 6 sizes: 0.5B, 1.5B, 3B, 7B, 14B and 32B. There are significant 
improvements in code generation, code reasoning and code fixing. The 32B model has competitive performance with 
OpenAI’s GPT-4o. ... Code Generation: Qwen2.5 Coder 32B Instruct, ...

[qwen / qwen2.5-coder-32b-instruct](https://docs.api.nvidia.com/nim/reference/qwen-qwen2_5-coder-32b-instruct)
As of now, Qwen2.5-Coder has covered six mainstream model sizes, 0.5, 1.5, 3, 7, 14, 32 billion parameters, to meet
the needs of different developers. Qwen2.5-Coder brings the following improvements upon CodeQwen1.5: Significant 
improvements ...

[GitHub - inferless/qwen2.5-coder-32b-instruct: A State-Of-The-Art coder LLM, tailored for instruction-based tasks,
particularly in code generation, reasoning, and repair. gpu: A100 | collections: ["vLLM"] · 
GitHub](https://github.com/inferless/qwen2.5-coder-32b-instruct)
A State-Of-The-Art coder LLM, tailored for instruction-based tasks, particularly in code generation, reasoning, and
repair. gpu: A100 | collections: ["vLLM"] </metada...

Out: None

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Number of parameters in billions                                                                               
  params_billions = 32                                                                                             
                                                                                                                   
  # Convert to number of parameters                                                                                
  params = params_billions * 10**9                                                                                 
                                                                                                                   
  # Memory calculation in bytes (2 bytes per parameter for float16)                                                
  memory_bytes = params * 2                                                                                        
                                                                                                                   
  # Convert to GB (1 GB = 1024^3 bytes)                                                                            
  memory_gb = memory_bytes / (1024**3)                                                                             
                                                                                                                   
  print(f"Number of parameters: {params:,}")                                                                       
  print(f"Memory required: {memory_bytes:,} bytes")                                                                
  print(f"Memory required: {memory_gb:.2f} GB")                                                                    
                                                                                                                   
  final_answer(memory_gb)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Number of parameters: 32,000,000,000
Memory required: 64,000,000,000 bytes
Memory required: 59.60 GB

Final answer: 59.604644775390625

[Step 2: Duration 3.52 seconds| Input tokens: 6,020 | Output tokens: 342]


RISPOSTA FINALE:
59.604644775390625


### Query 3: Test libero

Prova a fare una domanda a tua scelta! L'agente ha a disposizione:
- Ricerca web (può trovare informazioni aggiornate)
- Orologio mondiale
- Calcolatrice

In [ ]:
# Modifica questa query con la tua domanda!
mia_domanda = "Qual è la capitale della Nuova Zelanda e che ore sono lì adesso?"

risposta = agent.run(mia_domanda)

print("\n" + "="*50)
print("RISPOSTA FINALE:")
print(risposta)

## 9. Interfaccia Web con Gradio (Opzionale)

`smolagents` include un'integrazione con **Gradio** per creare una chat UI nel browser con una sola riga di codice.

> **Nota:** questa cella lancia un server locale. Apparirà un link `http://127.0.0.1:7860`. Fermalo con il pulsante `■ Stop` nella toolbar del notebook o con `Kernel → Interrupt`.

In [9]:
from smolagents import GradioUI

# Lancia l'interfaccia chat nel browser
# Interrompi l'esecuzione della cella per fermare il server
GradioUI(agent).launch()

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://44d306ee20c95f19ba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Qual'è la capitale della Nuova Zelanda e che ore sono lì adesso?                                                │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I need to find the capital of New Zealand and the current time in that capital city.                               
                                                                                                                   
First, let me find the capital of New Zealand:                                                                     
<code>                                                                                                             
capital = web_search("capital of New Zealand")                                                                     
print(f"Capital of New Zealand: {capital}")                                                                        
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  capital = web_search("capital of New Zealand")                                                                   
  print(f"Capital of New Zealand: {capital}")                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I have all the information needed to answer the question. The capital of New Zealand is Wellington, and the current
time there is 2026-05-11 04:25:00.                                                                                 
                                                                                                                   
Let me provide the final answer:                                                                                   
<code>                                                                                                             
final_answer("The capital of New Zealand is Wellington, and the current time there is 2026-05-11 04:25:00.")       
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The capital of New Zealand is Wellington, and the current time there is 2026-05-11 04:25:00.")     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The capital of New Zealand is Wellington, and the current time there is 2026-05-11 04:25:00.

[Step 7: Duration 1.54 seconds| Input tokens: 22,388 | Output tokens: 688]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://44d306ee20c95f19ba.gradio.live


## 10. Aggiungere un Nuovo Tool — Esercizio

Ora tocca a te! Completa il tool qui sotto e aggiungilo all'agente.

Idee:
- Un tool che converte valute (es. EUR → USD)
- Un tool che conta le parole in un testo
- Un tool che genera una battuta casuale

In [ ]:
@tool
def my_tool(input_text: str) -> str:  # Modifica i tipi se necessario
    """Descrivi cosa fa il tuo tool.
    Args:
        input_text: Descrivi questo argomento.
    """
    # Scrivi qui la tua logica
    return f"Il tuo tool ha ricevuto: {input_text}"


# Ricrea l'agente con il nuovo tool
agent_con_nuovo_tool = CodeAgent(
    model=model,
    tools=[search_tool, get_current_time_in_timezone, calculate, my_tool],
    max_steps=5,
    verbosity_level=1,
)

print(f"Tool disponibili: {[t.name for t in agent_con_nuovo_tool.tools.values()]}")

## Riepilogo

In questo notebook abbiamo:

| Passo | Cosa abbiamo fatto |
|-------|-------------------|
| Tool custom | Creato tool con `@tool`, type hints e docstring |
| Tool built-in | Usato `DuckDuckGoSearchTool` di smolagents |
| Modello | Connesso a `Qwen2.5-Coder-32B` via API HuggingFace |
| Agente | Costruito un `CodeAgent` con più tool |
| Esecuzione | Osservato il ciclo Think → Act → Observe |
| UI | Lanciato un'interfaccia chat con Gradio |